In [100]:
import pandas as pd
import numpy as np
import random

In [101]:
from typing import Callable

class MyLogReg:
    def __init__(self, n_iter: int = 10, learning_rate: float = 0.1,
                metric = None, reg = None, l1_coef = 0, l2_coef = 0,
                sgd_sample=None, random_state=42
                ):
        self.n_iter = n_iter
        self.learning_rate = learning_rate
        self.metric = metric
        self.reg = reg
        self.l1_coef = l1_coef
        self.l2_coef = l2_coef

        self.weights = None
        self.eps = 1e-15
        self.__metric_value = None
        self.sgd_sample = sgd_sample
        self._seed(random_state)

    def _seed(self, random_state):
        self.random_state = random_state 
        random.seed(self.random_state)

    def __str__(self):
        return f"MyLogReg class: n_iter={self.n_iter}, learning_rate={self.learning_rate}"

    def _accuracy(self, y_true, y_pred) -> float:
        return np.mean(y_true == y_pred)

    def _precision(self, y_true, y_pred) -> float:
        mask = y_pred == 1
        return y_true[mask].mean()

    def _recall(self, y_true, y_pred) -> float:
        mask = y_true == 1
        return y_pred[mask].mean()

    def _f1(self, y_true, y_pred) -> float:
        precision = self._precision(y_true, y_pred)
        recall = self._recall(y_true, y_pred)
        f1 = 2 * precision * recall / (precision + recall)
        return f1

    def _roc_auc(self, y_true: pd.Series, y_pred: pd.Series):
        y_true = y_true.copy()
        y_true.name = 'true'
        y_pred = y_pred.copy()
        y_pred.name = 'pred'
        y_pred = np.round(y_pred, 10)
        
        y = pd.concat([y_pred, y_true], axis=1).sort_values(["pred", "true"], ascending=False)    

        roc_auc = 0 
        for i in range(len(y)):
            if y['true'].iloc[i] == 0:
                p = y['pred'].iloc[i]
                for j in range(i):
                    if y['true'].iloc[j] == 1 and y['pred'].iloc[j] > p :
                        roc_auc += 1
                    elif y['true'].iloc[j] == 1 and y['pred'].iloc[j] == p:
                        roc_auc += 0.5
        roc_auc /= np.sum(y['true']) * (len(y['true']) - np.sum(y['true']))

        return roc_auc
    
    def log_loss(self, y_true, y_pred):
        return - np.mean(y_true * np.log(y_pred + self.eps) +  (1 - y_true) * np.log(1 - y_pred + self.eps)) 

    def _get_metric_value(self, y_true, y_pred):
        if self.metric != 'roc_auc':
            y_pred = y_pred > 0.5
        if self.metric == 'accuracy':
            return self._accuracy(y_true, y_pred)
        if self.metric == 'precision':
            return self._precision(y_true, y_pred)
        if self.metric == 'recall':
            return self._recall(y_true, y_pred)
        if self.metric == 'f1':
            return self._f1(y_true, y_pred)
        if self.metric == 'roc_auc':
            return self._roc_auc(y_true, y_pred)
    
    def fit(self, X: pd.DataFrame, y: pd.Series, verbose: int = 10):
        
        if '__bias' in X.columns:
            raise Exception("Column '__bias' exists in X")
        
        X = X.copy()

        X.insert(loc=0, column='__bias', value=1)

        count_features = len(X.columns)
        n = len(X)
            
        self.weights = pd.Series(1, X.columns)

        for i in range(1, self.n_iter + 1):
            
            y_pred = 1 / (1 + np.exp(-X.dot(self.weights)))
            log_loss = self.log_loss(y, y_pred)

            if self.reg == 'l1':
                log_loss += self.l1_coef * np.sum(np.abs(self.weights))

            if self.reg == 'l2':
                log_loss += self.l2_coef * np.sum(np.square(self.weights))

            if self.reg == 'elasticnet':
                log_loss += self.l1_coef * np.sum(np.abs(self.weights)) + self.l2_coef * np.sum(np.square(self.weights))
            
            if self.sgd_sample is not None:
                if isinstance(self.sgd_sample, int):
                    sgd_sample = self.sgd_sample
                elif isinstance(self.sgd_sample, float):
                    sgd_sample = round(self.sgd_sample * n)
                    
                sample_rows_idx = random.sample(range(X.shape[0]), sgd_sample)
            
                X_hat = []
                y_hat = []
                y_pred_hat = []
                indexes_hat = []
                for j in sample_rows_idx:
                    X_hat.append(X.iloc[j])
                    y_hat.append(y.iloc[j])
                    y_pred_hat.append(y_pred.iloc[j])
                    indexes_hat.append(X.index[j])
                X_hat = pd.concat(X_hat, axis=1).T
                y_hat = pd.Series(y_hat, index= indexes_hat)
                y_pred_hat = pd.Series(y_pred_hat, index= indexes_hat)
            else:
                X_hat = X
                y_hat = y
                y_pred_hat = y_pred
            

            gradient = (y_pred_hat - y_hat).dot(X_hat) / len(X_hat)

            if self.reg == 'l1':
                gradient += self.l1_coef * np.sign(self.weights)

            if self.reg == 'l2':
                gradient += self.l2_coef * self.weights * 2

            if self.reg == 'elasticnet':
                gradient += self.l1_coef * np.sign(self.weights) + self.l2_coef * self.weights * 2            
            
            if isinstance(self.learning_rate, float):
                learning_rate = self.learning_rate
            
            if isinstance(self.learning_rate, Callable):
                learning_rate = self.learning_rate(i)
            

            self.weights -= learning_rate * gradient

        y_pred = 1 / (1 + np.exp(-X.dot(self.weights)))
        self.__metric_value = self._get_metric_value(y, y_pred)

    def get_coef(self) -> np.ndarray:
        if self.weights is None:
            return None

        return self.weights.values[1:]

    def predict_proba(self, X: pd.DataFrame) -> pd.Series:
        X = X.copy()

        X.insert(loc=0, column='__bias', value=1)

        return 1 / (1 + np.exp(-X.dot(self.weights)))

    def predict(self, X: pd.DataFrame) -> pd.Series:
        y_pred = self.predict_proba(X)
        return y_pred > 0.5

    def get_best_score(self):
        return self.__metric_value
            